# External Tool Research Agent — Databricks Showcase

**Purpose:** Demonstrate how a Databricks-hosted LLM can use external tools, answer research questions, and expose its execution through MLflow.

> **Presentation note:** This notebook is designed for a live capability demo. Explain the business value in the Markdown cells; execute the code cells to show the real behaviour.

## What this showcases

- **Databricks Model Serving:** access an LLM through a serving endpoint.
- **LangChain tools and agent:** connect external search tools and let the agent choose when to use them.
- **MLflow tracing:** inspect model/tool execution, inputs, outputs, and errors.
- **Evaluation (extension):** create a repeatable test set and compare agent versions.

**Architecture:** User question → Databricks-hosted model → agent/tool selection → external source(s) → synthesized response → MLflow trace.

### Install dependencies

Run this cell once, then restart Python if Databricks asks you to. If your environment already has compatible packages, you can skip installation.

In [0]:
%pip install -U -q langchain langchain-core langgraph langgraph-prebuilt langchain-community databricks-langchain ddgs wikipedia youtube-search
%pip install --upgrade -q langchain langchain-core langgraph langgraph-prebuilt
dbutils.library.restartPython()

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.
Looking in indexes: [REDACTED]
  Attempting uninstall: langgraph-sdk
    Found existing installation: langgraph-sdk 0.3.15
    Uninstalling langgraph-sdk-0.3.15:
      Successfully uninstalled langgraph-sdk-0.3.15
  Attempting uninstall: langgraph-prebuilt
    Found existing installation: langgraph-prebuilt 1.0.13
    Uninstalling langgraph-prebuilt-1.0.13:
      Successfully uninstalled langgraph-prebuilt-1.0.13
  Attempting uninstall: langgraph
    Found existing installation: langgraph 1.0.10
    Uninstalling langgraph-1.0.10:
      Successfully uninstalled langgraph-1.0.10
  Attempting uninstall: langchain
    Found existing installation: langchain 1.2.10
    Uninstalling langchain-1.2.10:
      Successfully uninstalled langchain-1.2.10
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4/4 [langchain]
ERROR: pip's dependency resolver does not currently take into acco

## 0 — Workspace prerequisites

Before running:

1. In your Databricks workspace, confirm a chat model serving endpoint is available to you.
2. Use the endpoint name below (not its URL).
3. Install the Python packages in the next cell if your cluster/serverless environment permits it.
4. External search tools require internet access from the compute environment. If outbound access is restricted, the relevant tool may fail.

This notebook uses a **model endpoint name placeholder** intentionally—endpoint names differ between workspaces.

In [0]:
# Update this to an endpoint available in YOUR Databricks workspace.
MODEL_ENDPOINT = "databricks-meta-llama-3-3-70b-instruct"

# Optional: inspect your current environment
import sys
print("Python:", sys.version)

Python: 3.12.3 (main, Aug 31 2026, 10:18:26) [GCC 13.3.0]


## 1 — Databricks capability: Foundation Model access

**What to say:** Databricks Model Serving gives applications a managed endpoint for model inference. LangChain's `ChatDatabricks` is the integration layer we use in this notebook; the model itself is served by Databricks.

We first ask the model a question **without tools**. This establishes a baseline.

In [0]:
from databricks_langchain import ChatDatabricks

llm = ChatDatabricks(
    endpoint=MODEL_ENDPOINT,
    temperature=0,
)

baseline = llm.invoke("In one sentence, explain what a research assistant does.")
print(baseline.content)

A research assistant supports researchers and academics by collecting and organizing data, conducting literature reviews, preparing materials, and performing various administrative tasks to help facilitate studies, experiments, and projects.


## 2 — External tools: connect information sources

**What to say:** A model's built-in knowledge may be outdated or incomplete. Tools let an agent retrieve information at runtime. These tools are external integrations—not Databricks-native data sources.

We'll connect:
- Wikipedia for encyclopedia-style lookups
- DuckDuckGo for web search
- YouTube search for video discovery

In [0]:
from langchain_community.tools import WikipediaQueryRun
from langchain_community.utilities import WikipediaAPIWrapper
from langchain_community.tools import DuckDuckGoSearchRun
from langchain_community.tools.youtube.search import YouTubeSearchTool

wikipedia = WikipediaQueryRun(
    api_wrapper=WikipediaAPIWrapper(top_k_results=3, doc_content_chars_max=2500)
)
web_search = DuckDuckGoSearchRun()
youtube_search = YouTubeSearchTool()

tools = [wikipedia, web_search, youtube_search]

for tool in tools:
    print(f"{tool.name}: {tool.description[:180]}...")

wikipedia: A wrapper around Wikipedia. Useful for when you need to answer general questions about people, places, companies, facts, historical events, or other subjects. Input should be a sea...
duckduckgo_search: A wrapper around DuckDuckGo Search. Useful for when you need to answer questions about current events. Input should be a search query....
youtube_search: search for youtube videos associated with a person. the input to this tool should be a comma separated list, the first part contains a person name and the second a number that is t...


### 2A — Test tools directly

Before giving tools to an agent, test each integration independently. This helps distinguish a tool/network problem from an agent problem.

In [0]:
# Run these one at a time if an external service rate-limits requests.
# Each tool is tested independently so one failing integration does not stop the others.
try:
    print(wikipedia.run("Christopher Nolan"))
except Exception as e:
    print(f"wikipedia FAILED: {type(e).__name__}: {e}")

try:
    print(web_search.run("Christopher Nolan latest film"))
except Exception as e:
    print(f"web_search FAILED: {type(e).__name__}: {e}")

try:
    print(youtube_search.run("Christopher Nolan film interview"))
except Exception as e:
    print(f"youtube_search FAILED: {type(e).__name__}: {e}")

Page: Christopher Nolan
Summary: Sir Christopher Edward Nolan (born 30 July 1970) is a British and American filmmaker. Known for his Hollywood blockbusters with complex storytelling, Nolan is considered a leading filmmaker of the 21st century. His films have earned over $8 billion worldwide, making him the third-highest-grossing film director of all time. Nolan's accolades include two Academy Awards, a Golden Globe Award, and two British Academy Film Awards. He was also appointed a Commander of the British Empire (CBE) in 2019 and was knighted in 2024 for his contributions to film.
Nolan developed an interest in filmmaking from a young age. After studying English literature at University College London, he made several short films before his feature film debut with Following (1998). Nolan gained international recognition with his second film, Memento (2000), and transitioned into studio filmmaking with Insomnia (2002). He became a high-profile director with The Dark Knight trilogy (200

## 3 — Build the agent

**What to say:** An agent combines a model, instructions, and tools. The model can decide whether a tool is useful, call it, inspect its result, and continue before responding.

The agent is built with LangChain's `create_agent`; Databricks provides the model endpoint used by it.

In [0]:
from langgraph.runtime import ExecutionInfo
from langchain.agents import create_agent

SYSTEM_PROMPT = """You are a careful research assistant.
Use tools when the question needs current, factual, or source-specific information.
Do not claim that you searched or watched something unless a tool result supports it.
Clearly distinguish facts from uncertainty, and give a concise useful answer.
When useful, mention which source/tool informed your answer.
"""

agent = create_agent(
    model=llm,
    tools=tools,
    system_prompt=SYSTEM_PROMPT,
)

print("Agent initialized with:", [t.name for t in tools])

Agent initialized with: ['wikipedia', 'duckduckgo_search', 'youtube_search']


## 4 — Live demo: Movie Research Agent

**Demo prompt:** Ask about a movie/person and let the agent decide which tools to call.

Try:
- `Research Christopher Nolan: give me a short career overview and find a useful interview on YouTube.`
- `Find information about Dune: Part Two and suggest a video review.`
- `What is the background of the film Interstellar? Find a related YouTube result.`

The agent's answer depends on the live tool results; verify facts and links during the presentation.

In [0]:
from pprint import pprint

demo_question = (
    "Research Christopher Nolan. Give me a concise overview and find "
    "a relevant YouTube interview or video."
)

response = agent.invoke({
    "messages": [{"role": "user", "content": demo_question}]
})

# LangChain agent responses are message-based
pprint(response["messages"][-1].content)

('Christopher Nolan is a renowned British and American filmmaker known for his '
 'complex storytelling and Hollywood blockbusters. He has directed several '
 'successful films, including the Dark Knight trilogy, Inception, and '
 'Interstellar. Nolan has received numerous awards and accolades for his work, '
 'including two Academy Awards and a Golden Globe Award. He is also known for '
 'his use of practical special effects, experimental soundscapes, and '
 'large-format film photography. \n'
 '\n'
 'You can find a relevant YouTube interview with Christopher Nolan at '
 'https://www.youtube.com/watch?v=HfTbCet4qEw or '
 'https://www.youtube.com/watch?v=3jHpisxcmkc.')


## 5 — MLflow: trace the agent execution

**Databricks capability:** MLflow Tracing can help inspect an agent run, including model and tool operations, to understand execution and troubleshoot failures.

Enable LangChain autologging before the next agent call. Depending on the installed MLflow/LangChain versions, trace capture may require compatible versions or additional configuration.

In [0]:
import mlflow

mlflow.langchain.autolog()

traced_question = "Find a Wikipedia overview of the film Interstellar."
traced_response = agent.invoke({
    "messages": [{"role": "user", "content": traced_question}]
})

print(traced_response["messages"][-1].content)
print("After execution, open the MLflow experiment's Traces tab in the Databricks UI.")

The film Interstellar, directed by Christopher Nolan, is a 2014 epic science fiction drama that follows a group of astronauts as they travel through space in search of a new home for humanity. The film features an ensemble cast, including Matthew McConaughey, Anne Hathaway, and Jessica Chastain, and was shot on 35 mm film and IMAX 70 mm. It premiered in Los Angeles on October 26, 2014, and was released in theaters in the United States on November 5, 2014. The film received generally positive reviews from critics and was a commercial success, grossing $681.1 million worldwide during its initial theatrical run. It was also nominated for several awards, including five Academy Awards, and won the award for Best Visual Effects. The film's soundtrack, composed by Hans Zimmer, was also widely acclaimed and nominated for several awards.
After execution, open the MLflow experiment's Traces tab in the Databricks UI.


Trace(trace_id='tr-fc4744b2997c947894c86e9a745ee995')

## 6 — Compare different LLM models with MLflow

**Goal:** Compare multiple Databricks Model Serving endpoints using the same questions and settings. This isolates the model as the main variable.

For a fair comparison:
- Use the same evaluation questions and temperature for every model.
- Use endpoints that you are authorized to query.
- Keep the prompt and output instructions identical.
- Compare response quality after human review, alongside latency and reliability.

**What MLflow records:** one run per model, model/endpoint parameters, average and total latency, successful/failed question counts, and the generated answers as an artifact.

> Add at least two available model-serving endpoint names below. The first entry reuses the endpoint configured at the top of this notebook. Replace example endpoint names with real endpoints in your workspace. This section compares direct model responses (not agent tool selection); the existing agent tracing section remains unchanged.

In [0]:

import time
import json
import mlflow
import pandas as pd

from databricks_langchain import ChatDatabricks
from langchain.agents import create_agent
from IPython.display import display


# ==========================================================
# 1. MODEL ENDPOINTS
# ==========================================================

MODEL_ENDPOINTS = {
    "model_1": MODEL_ENDPOINT,
    "model_2": "databricks-gpt-oss-20b",
    "model_3": "databricks-qwen35-122b-a10b",
}


# ==========================================================
# 2. COMMON QUESTIONS
# ==========================================================

COMPARISON_QUESTIONS = [
    "Who is Christopher Nolan? Give a concise factual overview.",
    "Name three Christopher Nolan films and briefly describe their themes.",
    "Why is Interstellar considered a science-fiction film? Answer in 3-4 sentences.",
]

COMPARISON_TEMPERATURE = 0


# ==========================================================
# 3. MLFLOW CONFIGURATION
# ==========================================================

EXPERIMENT_NAME = (
    "/Users/kiritjain@cyntexa.com/"
    "External_Tool_Research_Agent_Model_Comparison"
)

mlflow.set_tracking_uri("databricks")
mlflow.set_experiment(EXPERIMENT_NAME)

# Enable LangChain tracing
mlflow.langchain.autolog()


# ==========================================================
# 4. HELPER: EXTRACT TOKEN USAGE
# ==========================================================

def extract_usage(message):

    usage = getattr(message, "usage_metadata", None) or {}
    
    if not usage:
        metadata = getattr(message, "response_metadata", {}) or {}
        usage = metadata.get("token_usage", {}) or {}

    return {
        "input_tokens": usage.get(
            "input_tokens", usage.get("prompt_tokens")
        ),
        "output_tokens": usage.get(
            "output_tokens", usage.get("completion_tokens")
        ),
        "total_tokens": usage.get("total_tokens"),
    }


# ==========================================================
# 5. RUN AND TRACE AN AGENT
# ==========================================================

@mlflow.trace(name="research_agent_comparison")
def traced_agent_call(endpoint_name, question):

    model = ChatDatabricks(
        endpoint=endpoint_name,
        temperature=COMPARISON_TEMPERATURE,
    )

    agent = create_agent(
        model=model,
        tools=tools,
        system_prompt=SYSTEM_PROMPT,
    )

    response = agent.invoke({
        "messages": [
            {"role": "user", "content": question}
        ]
    })

    messages = response.get("messages", [])

    tool_call_details = []
    tool_results = []
    input_tokens = 0
    output_tokens = 0
    total_tokens = 0

    for message in messages:

        # Capture requested tool calls
        calls = getattr(message, "tool_calls", []) or []

        for call in calls:
            tool_call_details.append({
                "tool_name": call.get("name"),
                "tool_input": call.get("args"),
            })

        # Capture actual tool execution results
        if message.__class__.__name__ == "ToolMessage":

            tool_results.append({
                "tool_name": getattr(message, "name", None),
                "tool_output": str(message.content),
            })

        # Capture token usage from model messages
        if message.__class__.__name__ in ("AIMessage", "AIMessageChunk"):

            usage = extract_usage(message)

            if usage["input_tokens"] is not None:
                input_tokens += usage["input_tokens"]

            if usage["output_tokens"] is not None:
                output_tokens += usage["output_tokens"]

            if usage["total_tokens"] is not None:
                total_tokens += usage["total_tokens"]

    # Get the final answer
    answer = ""

    for message in reversed(messages):
        if message.__class__.__name__ == "AIMessage":
            if not getattr(message, "tool_calls", []):
                answer = message.content
                break

    if not answer and messages:
        answer = messages[-1].content

    return {
        "answer": str(answer),
        "tool_calls": tool_call_details,
        "tool_results": tool_results,
        "input_tokens": input_tokens or None,
        "output_tokens": output_tokens or None,
        "total_tokens": total_tokens or None,
    }


# ==========================================================
# 6. MODEL COMPARISON
# ==========================================================

detailed_results = []
comparison_results = []

for model_label, endpoint_name in MODEL_ENDPOINTS.items():

    if not endpoint_name or "YOUR_" in endpoint_name or "<" in endpoint_name:
        print(f"Skipping {model_label}: invalid endpoint.")
        continue

    print(f"\nComparing agent: {model_label} ({endpoint_name})")

    latencies = []
    successful = 0
    failed = 0

    with mlflow.start_run(run_name=model_label):

        mlflow.log_params({
            "model_label": model_label,
            "endpoint_name": endpoint_name,
            "temperature": COMPARISON_TEMPERATURE,
            "question_count": len(COMPARISON_QUESTIONS),
            "tools_available": ", ".join(t.name for t in tools),
        })

        for question_number, question in enumerate(
            COMPARISON_QUESTIONS, start=1
        ):

            start_time = time.perf_counter()

            try:

                result = traced_agent_call(endpoint_name, question)

                elapsed = time.perf_counter() - start_time
                latencies.append(elapsed)
                successful += 1

                tool_names = list(dict.fromkeys(
                    item["tool_name"]
                    for item in result["tool_calls"]
                    if item["tool_name"]
                ))

                detailed_results.append({
                    "model": model_label,
                    "endpoint": endpoint_name,
                    "question_number": question_number,
                    "question": question,
                    "answer": result["answer"],
                    "tools_called": ", ".join(tool_names),
                    "tool_call_count": len(result["tool_calls"]),
                    "tool_call_details": json.dumps(
                        result["tool_calls"], default=str
                    ),
                    "tool_results": json.dumps(
                        result["tool_results"], default=str
                    ),
                    "input_tokens": result["input_tokens"],
                    "output_tokens": result["output_tokens"],
                    "total_tokens": result["total_tokens"],
                    "latency_seconds": round(elapsed, 3),
                    "status": "success",
                    "error": None,
                })

            except Exception as exc:

                elapsed = time.perf_counter() - start_time
                failed += 1

                detailed_results.append({
                    "model": model_label,
                    "endpoint": endpoint_name,
                    "question_number": question_number,
                    "question": question,
                    "answer": "",
                    "tools_called": "",
                    "tool_call_count": 0,
                    "tool_call_details": "[]",
                    "tool_results": "[]",
                    "input_tokens": None,
                    "output_tokens": None,
                    "total_tokens": None,
                    "latency_seconds": round(elapsed, 3),
                    "status": "failed",
                    "error": f"{type(exc).__name__}: {exc}",
                })

                print(f"Request failed: {exc}")

        # --------------------------------------------------
        # MODEL SUMMARY
        # --------------------------------------------------

        model_rows = [
            row for row in detailed_results
            if row["model"] == model_label
        ]

        successful_rows = [
            row for row in model_rows
            if row["status"] == "success"
        ]

        all_tools = []

        for row in successful_rows:
            if row["tools_called"]:
                all_tools.extend(
                    name.strip()
                    for name in row["tools_called"].split(",")
                )

        summary = {
            "model": model_label,
            "endpoint": endpoint_name,
            "successful": successful,
            "failed": failed,
            "average_latency_seconds": (
                round(sum(latencies) / len(latencies), 3)
                if latencies else None
            ),
            "unique_tools_used": ", ".join(sorted(set(all_tools))),
            "total_tool_calls": sum(
                row["tool_call_count"] for row in successful_rows
            ),
            "average_input_tokens": pd.to_numeric(
                [r["input_tokens"] for r in successful_rows],
                errors="coerce"
            ).mean(),
            "average_output_tokens": pd.to_numeric(
                [r["output_tokens"] for r in successful_rows],
                errors="coerce"
            ).mean(),
            "average_total_tokens": pd.to_numeric(
                [r["total_tokens"] for r in successful_rows],
                errors="coerce"
            ).mean(),
        }

        comparison_results.append(summary)

        mlflow.log_metrics({
            "successful_questions": successful,
            "failed_questions": failed,
            "total_tool_calls": summary["total_tool_calls"],
        })

        if latencies:
            mlflow.log_metric(
                "average_latency_seconds",
                summary["average_latency_seconds"]
            )

        print(
            f"Successful: {successful} | Failed: {failed}"
        )

        print("Tools used:", summary["unique_tools_used"])


# ==========================================================
# 7. CREATE DATAFRAMES
# ==========================================================

detailed_df = pd.DataFrame(detailed_results)

comparison_df = pd.DataFrame(comparison_results)


# ==========================================================
# 8. DISPLAY RESULTS
# ==========================================================

print("\n========== DETAILED AGENT RESULTS ==========")
display(detailed_df)

print("\n========== AGENT COMPARISON SUMMARY ==========")
display(comparison_df)


# ==========================================================
# 9. SAVE RESULTS
# ==========================================================

mlflow.log_table(
    detailed_df,
    artifact_file="agent_detailed_results.json"
)

mlflow.log_table(
    comparison_df,
    artifact_file="agent_comparison_summary.json"
)

print("\nAgent comparison completed.")


If you are using MLflow Tracing, you can migrate your traces to Unity Catalog for unlimited storage, fine-grained access controls, and queryability from notebooks, SQL, and dashboards. Learn more: https://docs.databricks.com/aws/en/mlflow3/genai/tracing/migrate-traces-to-uc



Comparing agent: model_1 (databricks-meta-llama-3-3-70b-instruct)
Successful: 3 | Failed: 0
Tools used: wikipedia

Comparing agent: model_2 (databricks-gpt-oss-20b)
Successful: 3 | Failed: 0
Tools used: wikipedia

Comparing agent: model_3 (databricks-qwen35-122b-a10b)


/local_disk0/.ephemeral_nfs/envs/pythonEnv-b4be1193-bddf-4afe-b632-897287d92e08/lib/python3.12/site-packages/pydantic/main.py:464: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `str` - serialized value may not be as expected [field_name='content', input_value=[{'type': 'reasoning', 's...ation about people."}]}], input_type=list])
  return self.__pydantic_serializer__.to_python(
/local_disk0/.ephemeral_nfs/envs/pythonEnv-b4be1193-bddf-4afe-b632-897287d92e08/lib/python3.12/site-packages/pydantic/main.py:464: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `str` - serialized value may not be as expected [field_name='content', input_value=[{'type': 'reasoning', 's...narrative structures."}], input_type=list])
  return self.__pydantic_serializer__.to_python(
/local_disk0/.ephemeral_nfs/envs/pythonEnv-b4be1193-bddf-4afe-b632-897287d92e08/lib/python3.12/site-packages/pydantic/main.py:464: UserWarning: Pyda

Successful: 3 | Failed: 0
Tools used: wikipedia

========== DETAILED AGENT RESULTS ==========


,model,endpoint,question_number,question,answer,tools_called,tool_call_count,tool_call_details,tool_results,input_tokens,output_tokens,total_tokens,latency_seconds,status,error
0,model_1,databricks-meta-llama-3-3-70b-instruct,1,Who is Christopher Nolan? Give a concise factu...,Christopher Nolan is a British and American fi...,wikipedia,1,"[{""tool_name"": ""wikipedia"", ""tool_input"": {""qu...","[{""tool_name"": ""wikipedia"", ""tool_output"": ""Pa...",2657,136,2793,3.949,success,None
1,model_1,databricks-meta-llama-3-3-70b-instruct,2,Name three Christopher Nolan films and briefly...,Three notable Christopher Nolan films are:\n\n...,,0,[],[],1055,179,1234,2.840,success,None
2,model_1,databricks-meta-llama-3-3-70b-instruct,3,Why is Interstellar considered a science-ficti...,Interstellar is considered a science-fiction f...,,0,[],[],1062,97,1159,1.612,success,None
3,model_2,databricks-gpt-oss-20b,1,Who is Christopher Nolan? Give a concise factu...,**Christopher Nolan** (Sir Christopher Edward ...,wikipedia,1,"[{""tool_name"": ""wikipedia"", ""tool_input"": {""qu...","[{""tool_name"": ""wikipedia"", ""tool_output"": ""Pa...",1243,449,1692,3.577,success,None
4,model_2,databricks-gpt-oss-20b,2,Name three Christopher Nolan films and briefly...,**1. Memento (2000)** \n*Theme:* The film exp...,wikipedia,1,"[{""tool_name"": ""wikipedia"", ""tool_input"": {""qu...","[{""tool_name"": ""wikipedia"", ""tool_output"": ""Pa...",1265,333,1598,3.787,success,None
5,model_2,databricks-gpt-oss-20b,3,Why is Interstellar considered a science-ficti...,Interstellar is classified as science‑fiction ...,,0,[],[],363,203,566,1.473,success,None
6,model_3,databricks-qwen35-122b-a10b,1,Who is Christopher Nolan? Give a concise factu...,"Sir Christopher Edward Nolan (born July 30, 19...",wikipedia,1,"[{""tool_name"": ""wikipedia"", ""tool_input"": {""qu...","[{""tool_name"": ""wikipedia"", ""tool_output"": ""Pa...",1852,421,2273,5.314,success,None
7,model_3,databricks-qwen35-122b-a10b,2,Name three Christopher Nolan films and briefly...,"[{""type"": ""reasoning"", ""summary"": [{""type"": ""s...",,0,[],[],594,400,994,3.552,success,None
8,model_3,databricks-qwen35-122b-a10b,3,Why is Interstellar considered a science-ficti...,"[{""type"": ""reasoning"", ""summary"": [{""type"": ""s...",,0,[],[],601,300,901,2.898,success,None



========== AGENT COMPARISON SUMMARY ==========


,model,endpoint,successful,failed,average_latency_seconds,unique_tools_used,total_tool_calls,average_input_tokens,average_output_tokens,average_total_tokens
0,model_1,databricks-meta-llama-3-3-70b-instruct,3,0,2.800,wikipedia,1,1591.333333,137.333333,1728.666667
1,model_2,databricks-gpt-oss-20b,3,0,2.946,wikipedia,2,957.000000,328.333333,1285.333333
2,model_3,databricks-qwen35-122b-a10b,3,0,3.921,wikipedia,1,1015.666667,373.666667,1389.333333



Agent comparison completed.


[Trace(trace_id='tr-e20ae5abd5de8166b47e0072d8ce3815'), Trace(trace_id='tr-79b845e4feb77d462f78254fd14ce382'), Trace(trace_id='tr-14963dc814c7ab96d174dabb8fdbd1a6'), Trace(trace_id='tr-1e7383a7ef574d0aa194862387017526'), Trace(trace_id='tr-0e74d62dd89f94de92e09ce0c886d386'), Trace(trace_id='tr-6d88dfe5e5d2d6cb27c3e8b10c346bca'), Trace(trace_id='tr-d40e7115803ac495116b2434839903df'), Trace(trace_id='tr-6544f9b33137510857c0e71c9424be64'), Trace(trace_id='tr-4dcbd2638689b914216c68afb8484e17')]

### How to read the model-comparison results
1. Open the External_Tool_Research_Agent_Model_Comparison experiment in MLflow.
2. Open Runs and select the model runs you want to compare.
3. Compare average_latency_seconds, successful_questions, and failed_questions.
4. Open each run's artifacts and inspect model_answers.json.
5. Review answer correctness, relevance, completeness, and factual support using the same rubric for every model.

**Important:** Latency and successful-call counts are measured automatically here. Correctness and relevance are not automatically scored by this code; review the saved answers or add a consistent MLflow GenAI evaluation dataset and scorers as a separate step. A faster model is not necessarily a better model.

## 7 — Inspect traces in the Databricks UI

After the preceding run:

1. Open **Experiments** (or the MLflow experiment configured for your workspace).
2. Open the relevant experiment.
3. Go to **Traces**.
4. Select a trace and inspect its inputs, outputs, model call, tool calls, timing, and any errors available in the trace.

**Presentation takeaway:** The answer is only one part of an agent system. Tracing helps developers understand *how* the agent produced it.

## 8 — Evaluation: how would we measure quality?

A polished production workflow should test more than whether the agent returns text. Create a small dataset of representative questions and check:

- Did it use a suitable tool?
- Are factual claims supported by tool results?
- Is the response relevant and complete?
- Does it handle tool failures gracefully?
- Is latency acceptable?

For a live presentation, show the MLflow evaluation UI or a small manually reviewed test set. Avoid claiming that a single successful response proves reliability.

## 9 — Databricks platform capabilities: what is native and what is integrated?

| Component | Role |
|---|---|
| Databricks Model Serving | Hosts/serves the selected model endpoint |
| `ChatDatabricks` | LangChain connector to the Databricks model endpoint |
| LangChain `create_agent` | Agent orchestration and tool calling |
| Wikipedia / DuckDuckGo / YouTube | External tools and services |
| MLflow Tracing | Observability of agent execution |
| MLflow Evaluation | Repeatable quality evaluation |
| Unity Catalog | Governance and controlled access to enterprise assets (not required for these public search tools) |

**Important distinction:** This is a Databricks-based showcase, not a claim that every library or external search service is built by Databricks.

## 10 — Troubleshooting checklist

- **Endpoint not found / permission error:** confirm `MODEL_ENDPOINT` and your permission to query it.
- **Package import error:** rerun `%pip` and restart Python; check package compatibility.
- **Wikipedia / DuckDuckGo / YouTube error:** test that tool directly; external network access or service limits may be the cause.
- **Agent does not call a tool:** make the request require information the tool can retrieve; inspect the trace and tool descriptions.
- **No MLflow trace:** check autologging compatibility and the workspace's MLflow tracing configuration.
- **Slow response:** external tools and model inference add latency; keep demo questions focused.

## 11 — Closing message

**What this demonstrates:** Databricks can be used as the model-serving and observability foundation for an agent application, while frameworks and external tools provide orchestration and live information retrieval.

**Next extension:** add a LangGraph workflow, a curated evaluation dataset, or a governed enterprise data tool.